# 06 – v2 Setup and Smoke Test

Run this **once per machine**, before any v2 stage (`RESULTS_AND_PLAN.md`, Part B). It checks that the environment, data and manifests are in place. It then runs the full evidence pipeline on 3 dev images per detector and verifies the parts that could not be tested on a laptop: batched forwards on the real detectors, GradCAM with captum, and the VLM Yes/No scoring. Finally it turns the measured timings into a GPU budget for the pilot and test stages.

| Section | Needs | Typical time |
|---|---|---|
| 1 Environment | GPU node | 2–5 min (incl. fast tests) |
| 2 Data | — | seconds (download: ~20 min) |
| 3 Manifests | legacy `results/` copied over | seconds |
| 4 Smoke run | GPU | 5–20 min (Faster R-CNN is slow) |
| 5 Real-model checks | GPU in this kernel | 5–10 min |
| 6 VLM smoke | GPU, Qwen weights (~16 GB) | 5 min |
| 7 Maps | — | seconds |

Every command goes through `cluster/task.sh`, the same file the cluster uses, so the smoke run exercises the exact production commands. Set `DRY_RUN = True` to print commands without running them.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src import notebook_utils as nbu

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

ENV_FILE = "cluster/cluster.env"   # created from cluster.env.example if missing - edit it first
DRY_RUN = False                    # True: print commands only
RUN_DOWNLOAD = False               # True: run scripts/download_data.sh when COCO is missing
DETECTOR_KEYS = ["yolox", "frcnn"]
SMOKE_IMAGES = "3"                 # images per detector for the smoke run

env = nbu.load_cluster_env(ENV_FILE)
PY = nbu.python_executable()
display(pd.Series(env, name="value").to_frame())

## 1. Environment

`cluster/env_check.sh` prints the GPU, the torch/CUDA build, and every required import. It uses `RUN_PREFIX` from the env file, so a container setup is checked as it will run. The two test commands then run the CPU suite and the model-dependent tests. These must pass here: they cover the GradCAM/captum and torchvision paths that were skipped on the development laptop.

In [ ]:
nbu.run_command(["bash", "cluster/env_check.sh", "--no-tests"], env={"AEDXAI_ENV": ENV_FILE}, dry_run=DRY_RUN)
nbu.run_command([PY, "-m", "pytest", "-q", "-m", "not slow", "tests"], dry_run=DRY_RUN)
nbu.run_command([PY, "-m", "pytest", "-q", "tests/test_detector.py", "tests/test_xai_methods.py"], dry_run=DRY_RUN)

## 2. Data

Required: COCO val2017 images, `instances_val2017.json` and the YOLOX-S weights. Optional: BDD100K (stage 4). The legacy `results/xai_selector_training_progress_*.csv` files are needed so the manifests can exclude the images the v1 selector was trained on.

In [ ]:
required = {
    "COCO images": env["COCO_IMAGES"],
    "COCO annotations": env["COCO_ANN"],
    "YOLOX-S weights": "data/checkpoints/yolox_s.pth",
    "legacy oracle CSV (yolox)": "results/xai_selector_training_progress_yolox-s.csv",
    "legacy oracle CSV (frcnn)": "results/xai_selector_training_progress_fasterrcnn_resnet50_fpn_v2.csv",
    "BDD100K images (optional)": env["BDD_IMAGES"],
    "BDD100K det_20 labels (optional)": env["BDD_LABELS"],
}
status = pd.DataFrame(
    [{"item": name, "path": path, "exists": (PROJECT_ROOT / path).exists()} for name, path in required.items()]
)
display(status)
if RUN_DOWNLOAD and not (PROJECT_ROOT / env["COCO_IMAGES"]).exists():
    nbu.run_command(["bash", "scripts/download_data.sh"], dry_run=DRY_RUN)

## 3. Manifests

The manifests fix disjoint splits (COCO: dev 300 / calib 1000 / test 2000; BDD100K: calib 300 / test 1000). **Commit `manifests/*.json` right after this cell**: every later result is defined by these lists. Re-running is a no-op while the files exist.

In [ ]:
coco_manifest = PROJECT_ROOT / env["MANIFEST"]
bdd_labels, bdd_ann = PROJECT_ROOT / env["BDD_LABELS"], PROJECT_ROOT / env["BDD_ANN"]
if bdd_labels.exists() and not bdd_ann.exists():
    nbu.run_task("setup-bdd-annotations", env_file=ENV_FILE, dry_run=DRY_RUN)
if not coco_manifest.exists():
    nbu.run_task("setup-manifests", env_file=ENV_FILE, dry_run=DRY_RUN)

for path in [coco_manifest, PROJECT_ROOT / env["BDD_MANIFEST"]]:
    if path.exists():
        manifest = json.loads(path.read_text())
        print(path.relative_to(PROJECT_ROOT), {k: len(v) for k, v in manifest["splits"].items()}, "excluded:", manifest["excluded"])
nbu.run_command(["git", "status", "--short", "manifests"], dry_run=DRY_RUN)

## 4. Smoke evidence run

Three dev images per detector with the **full** evidence config: all six methods, three controls, two shuffled maps, gradient features and the box protocol. The run directories are `results/evidence/smoke_dev_<detector>`. They are resumable, so re-running this cell only finishes missing images.

In [ ]:
for key in DETECTOR_KEYS:
    nbu.run_task(f"evidence-smoke-{key}", env_file=ENV_FILE, extra_env={"SMOKE_IMAGES": SMOKE_IMAGES}, dry_run=DRY_RUN)
SMOKE_RUNS = [nbu.run_name("dev", key, "smoke") for key in DETECTOR_KEYS]
display(nbu.evidence_progress(SMOKE_RUNS))

Inspect one explained detection. What to look for:
- `is_fallback` is False for the real methods.
- `ebpg_gt` is present when the detection is a TP.
- `bbox_prior` scores in the same range as the real methods on `composite`. That is the effect gate B measures.
- `explain_time` is largest for LIME and D-CLOSE.

In [ ]:
from src.evidence_io import iter_image_records

record = next(iter(iter_image_records([nbu.EVIDENCE_ROOT / SMOKE_RUNS[0]])))
det = next(d for d in record["detections"] if d.get("explained"))
print(record["image"], "| det", det["det_id"], det["class_name"], f"conf={det['confidence']:.2f}", "| GT label:", det["match"]["label"], det["match"]["fp_type"])
columns = ["composite", "ebpg", "oa", "oa_box", "sparsity", "ebpg_gt", "explain_time", "eval_time", "is_fallback"]
display(pd.DataFrame({m: {c: v.get(c) for c in columns} for m, v in det["methods"].items()}).T)
display(pd.Series(det.get("gradient") or {}, name="gradient features").to_frame().T)

### Budget projection

`s_per_explained_detection` is measured, so the projections below replace the estimates in `RESULTS_AND_PLAN.md` section B5. The lite config costs about 0.8× the full config. Faster R-CNN caps: pilot `NUM_IMAGES_dev_frcnn`, test `NUM_IMAGES_test_frcnn`, and `MAX_EXPLAINED=10` for test.

In [ ]:
timing = pd.DataFrame([nbu.timing_summary(run) for run in SMOKE_RUNS]).set_index("run")
display(timing.drop(columns=["s_per_method_per_detection"]))
display(pd.DataFrame({run: nbu.timing_summary(run)["s_per_method_per_detection"] for run in SMOKE_RUNS}))

plans = {
    "pilot yolox (dev, full, cap 20)": (SMOKE_RUNS[0], 300, 20, 1.0),
    "pilot frcnn (dev, full, cap 20)": (SMOKE_RUNS[1], int(env.get("NUM_IMAGES_dev_frcnn") or 300), 20, 1.0),
    "test yolox (lite, cap 10)": (SMOKE_RUNS[0], 2000, 10, 0.8),
    "test frcnn (lite, cap 10)": (SMOKE_RUNS[1], int(env.get("NUM_IMAGES_test_frcnn") or 2000), 10, 0.8),
}
display(pd.DataFrame({name: nbu.estimate_gpu_hours(run, n, cap, scale) for name, (run, n, cap, scale) in plans.items()}).T)

## 5. Real-model checks

These run in this kernel on the real detectors and one dev image:
- The evaluator gives the same curves with batch 11 as with batch 1.
- LIME and D-CLOSE give the same maps with batch 16 as with batch 1.
- Every CAM variant runs without falling back to the bbox prior (this needs captum).

Differences below the tolerance come from GPU kernels that pick different algorithms per batch size; they are not a bug. Anything above it is: stop, and do not start the pilot.

In [ ]:
import torch
import yaml

from src.detector import DetectorWrapper
from src.evaluator import AutoEvaluator
from src.provenance import load_image_list
from src.utils import load_image
from src.xai_methods import base_method_name, get_explainer

xai_cfg = yaml.safe_load((PROJECT_ROOT / "config/xai_config.yaml").read_text())["xai"]["methods"]
image_name = load_image_list(PROJECT_ROOT / env["MANIFEST"], split="dev")[0]
image = load_image(str(PROJECT_ROOT / env["COCO_IMAGES"] / image_name))
checks = []
for key in DETECTOR_KEYS:
    detector = DetectorWrapper(nbu.DETECTORS[key], config_path=str(PROJECT_ROOT / "config/detector_config.yaml"))
    detector.load_model()
    model, layer = detector.get_model(), detector.get_target_layer()
    detections = detector.detect(image)
    if not detections:
        checks.append({"detector": key, "check": "has detections", "max_abs_diff": None, "ok": False})
        continue
    det = detections[0]
    evaluator = AutoEvaluator(str(PROJECT_ROOT / "config/eval_config.yaml"))
    saliency = get_explainer("gcame", xai_cfg["gcame"]).explain(model, image, det, layer)
    single = evaluator.insertion_deletion_curves(saliency.map, model, image, det, batch_size=1)
    batched = evaluator.insertion_deletion_curves(saliency.map, model, image, det, batch_size=11)
    diff = max(
        float(np.abs(np.subtract(single["insertion_curve"], batched["insertion_curve"])).max()),
        float(np.abs(np.subtract(single["deletion_curve"], batched["deletion_curve"])).max()),
    )
    checks.append({"detector": key, "check": "evaluator batch 11 == batch 1", "max_abs_diff": diff, "ok": diff < 1e-3})
    for method, overrides in (("lime", {"num_perturbations": 64}), ("dclose", {"num_masks_dev": 48})):
        one = get_explainer(method, {**xai_cfg[method], **overrides, "batch_size": 1}).explain(model, image, det)
        many = get_explainer(method, {**xai_cfg[method], **overrides, "batch_size": 16}).explain(model, image, det)
        diff = float(np.abs(one.map - many.map).max())
        checks.append({"detector": key, "check": f"{method} batch 16 == batch 1", "max_abs_diff": diff, "ok": diff < 1e-2})
    for method in ("gradcam", "gradcam_raw", "gcame", "gcame_raw"):
        result = get_explainer(method, xai_cfg[base_method_name(method)]).explain(model, image, det, layer)
        ok = (not result.is_fallback) and result.map.shape == image.shape[:2]
        checks.append({"detector": key, "check": f"{method} runs without fallback", "max_abs_diff": None, "ok": ok})
    detector.unload_model()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
checks = pd.DataFrame(checks)
display(checks)
print("ALL CHECKS PASS" if bool(checks["ok"].all()) else "SOME CHECKS FAILED - fix before the pilot")

## 6. VLM smoke

The VLM pass reads the maps saved by the smoke run. `box_answer_mass` is P("Yes") + P("No") for the first answer token. A median well below 0.9 means the Yes/No token ids do not match the tokenizer, and the P(yes) features are then meaningless. On three images the TP/FP split is only a sanity check, not a result.

In [ ]:
from src.evidence_io import detections_frame

for key in DETECTOR_KEYS:
    nbu.run_task(f"vlm-smoke-{key}", env_file=ENV_FILE, dry_run=DRY_RUN)
frame = detections_frame([nbu.EVIDENCE_ROOT / run for run in SMOKE_RUNS], include_unexplained=False)
vlm_columns = [c for c in frame.columns if c.startswith("vlm__")]
if vlm_columns:
    display(frame.groupby(["detector", "is_tp"])[vlm_columns].median().round(3))
    print("median box_answer_mass:", round(float(frame["vlm__box_answer_mass"].median()), 3))
else:
    print("No VLM records yet.")

## 7. Look at the maps

Saved maps are overlaid in a single-hue blue ramp; orange is the detection box and white the matched GT box. Compare `gcame`, which has the Gaussian on the box, with `gcame_raw`, which has it removed. The difference is the box conditioning that the metric audit measures.

In [ ]:
import matplotlib.pyplot as plt

from src import v2_figures as vf

vf.apply_style()
run_dir = nbu.EVIDENCE_ROOT / SMOKE_RUNS[0]
for record in iter_image_records([run_dir]):
    npz = run_dir / "maps" / f"{Path(record['image']).stem}.npz"
    if not npz.exists():
        continue
    image = load_image(str(PROJECT_ROOT / env["COCO_IMAGES"] / record["image"]))
    for det in [d for d in record["detections"] if d.get("explained")][:2]:
        maps = vf.load_saved_maps(npz, det["det_id"], image.shape)
        title = f"{record['image']} · {det['class_name']} conf {det['confidence']:.2f} · {det['match']['label'].upper()}"
        vf.show_detection_maps(image, maps, det["bbox"], title=title, gt_bbox=det["match"].get("gt_bbox"))
        plt.show()
    break

## Checklist before the pilot

- [ ] Section 1: env check clean; fast and model-dependent tests pass.
- [ ] Section 3: `manifests/*.json` committed.
- [ ] Section 5: `ALL CHECKS PASS`.
- [ ] Section 6: median `box_answer_mass` ≥ 0.9.
- [ ] Budget table: the pilot fits. If Faster R-CNN does not, lower `NUM_IMAGES_dev_frcnn` in `cluster/cluster.env`.
- [ ] `git status` clean, so every shard manifest records `dirty: false`.

Next: `07_v2_run_experiments.ipynb`.